<a href="https://colab.research.google.com/github/aravindanmoorthy/Claude-Hackathon/blob/claude%2Fweather-alerts-parked-cars-58mry/safe_pilot_weather_alert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Safe Pilot — Weather Alert System
### USAA Hackathon Project

Checks real-time weather for GPS locations and determines whether a **Safe Pilot alert** should be triggered for a **parked vehicle**.

**API:** [Open-Meteo](https://open-meteo.com/) — Free, no API key required!

---
**Sections:**
- 🌍 **Section A** — Real live weather (5 locations)
- 🧪 **Section B** — Simulated scenarios with forced weather codes
  - Group 1: Currently clear → severe weather incoming (4 scenarios)
  - Group 2: Currently severe weather active (4 scenarios)
  - Group 3: All clear, no alert (1 scenario)
- ✏️ **Section C** — Test your own lat/lon

> Run each cell top to bottom using **Shift+Enter**

In [ ]:
# Install the requests library for making HTTP calls to the Open-Meteo API.
# In Google Colab, requests is usually pre-installed, but this ensures it's available.
!pip install requests --quiet
print('✅ Dependencies ready!')

In [ ]:
import requests       # For making HTTP GET calls to the Open-Meteo weather API
from dataclasses import dataclass  # For defining a clean structured WeatherAlert object
from datetime import datetime      # For formatting timestamps (e.g. clears_by time)
from copy import deepcopy          # For cloning the base API response in simulated scenarios
                                   # without modifying the original

print('✅ Imports successful!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# ALERT THRESHOLD CODES
# These are WMO (World Meteorological Organization) weather codes
# that represent conditions severe enough to trigger a Safe Pilot alert.
# Any current or forecasted hourly code in this set will fire an alert.
# ─────────────────────────────────────────────────────────────────
ALERT_THRESHOLD_CODES = {95, 96, 99, 82, 75, 65, 45, 48}

# ─────────────────────────────────────────────────────────────────
# SEVERE WEATHER CODES
# Full mapping of WMO weather codes to human-readable labels.
# Open-Meteo returns these codes in both current and hourly fields.
# Reference: https://open-meteo.com/en/docs#weathervariables
# ─────────────────────────────────────────────────────────────────
SEVERE_WEATHER_CODES = {
    0:  'Clear Sky',
    1:  'Mainly Clear',
    2:  'Partly Cloudy',
    3:  'Overcast',
    45: 'Foggy',
    48: 'Icy Fog',
    51: 'Light Drizzle',
    61: 'Light Rain',
    63: 'Moderate Rain',
    65: 'Heavy Rain',
    71: 'Light Snow',
    73: 'Moderate Snow',
    75: 'Heavy Snow',
    77: 'Snow Grains',
    80: 'Rain Showers',
    81: 'Heavy Showers',
    82: 'Violent Showers',
    85: 'Snow Showers',
    95: 'Thunderstorm',
    96: 'Thunderstorm + Hail',
    99: 'Thunderstorm + Heavy Hail',
}

# ─────────────────────────────────────────────────────────────────
# ALERT SEVERITY LEVELS
# Maps each severe weather code to a severity tier:
#   MEDIUM   → Inconvenient but manageable (fog, heavy rain)
#   HIGH     → Significant risk to vehicle and driver safety
#   CRITICAL → Immediate action required (thunderstorm + heavy hail)
# ─────────────────────────────────────────────────────────────────
ALERT_SEVERITY = {
    45: 'MEDIUM',   # Foggy
    48: 'HIGH',     # Icy Fog — ice buildup risk on windshield
    65: 'MEDIUM',   # Heavy Rain
    75: 'HIGH',     # Heavy Snow — ice and structural damage risk
    82: 'HIGH',     # Violent Showers — flash flood risk
    95: 'HIGH',     # Thunderstorm
    96: 'HIGH',     # Thunderstorm + Hail
    99: 'CRITICAL', # Thunderstorm + Heavy Hail — severe vehicle damage
}

# ─────────────────────────────────────────────────────────────────
# PARKED VEHICLE ADVICE
# Actionable guidance specific to a parked vehicle for each
# severe weather condition. Shown in the alert output to help
# the driver make a quick, informed decision.
# ─────────────────────────────────────────────────────────────────
PARKED_VEHICLE_ADVICE = {
    45: 'Move to covered garage — low visibility risk from other drivers.',
    48: 'Move vehicle to covered area to prevent ice buildup on windshield.',
    65: 'Move away from low-lying areas to avoid flood risk.',
    75: 'Move vehicle to a garage to prevent snow and ice damage.',
    82: 'Seek covered parking immediately — flash flood risk.',
    95: 'Move vehicle away from trees and open fields.',
    96: 'Move to covered parking NOW to prevent hail damage.',
    99: 'CRITICAL: Seek immediate covered shelter — severe hail damage risk.',
}

print('✅ Constants loaded!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# WeatherAlert — Core data model
#
# This dataclass holds everything needed to render a Safe Pilot
# alert for a given GPS location. It is populated by
# parse_weather_alert() after the API response is received.
# ─────────────────────────────────────────────────────────────────
@dataclass
class WeatherAlert:
    location_name:     str    # Human-readable name of the location
    lat:               float  # GPS latitude
    lon:               float  # GPS longitude
    is_severe_now:     bool   # True if current weather code is in ALERT_THRESHOLD_CODES
    condition_now:     str    # Human-readable current weather condition
    weather_code:      int    # Raw WMO weather code for current conditions
    wind_speed_mph:    float  # Current wind speed in mph
    visibility_miles:  float  # Current visibility converted from meters to miles
    temperature_f:     float  # Current temperature in Fahrenheit
    upcoming_alerts:   list   # List of severe conditions in the next 1 hour
    alert_required:    bool   # True if either current or upcoming conditions are severe
    severity:          str    # NONE / MEDIUM / HIGH / CRITICAL
    alert_message:     str    # Full alert message string shown to the driver
    vehicle_advice:    str    # Specific action advice for a parked vehicle
    starts_in_hours:   int    # Hours until bad weather begins (0 = already active)
    duration_hours:    int    # Estimated hours the severe weather will last
    clears_by:         str    # Human-readable time when conditions are expected to improve

print('✅ WeatherAlert data class defined!')

In [ ]:
def fetch_weather(lat: float, lon: float) -> dict:
    """
    Calls the Open-Meteo API for a given GPS coordinate and returns
    the raw JSON response containing current and hourly weather data.

    Parameters:
        lat (float): Latitude from GPS (e.g. 26.3017)
        lon (float): Longitude from GPS (e.g. -98.1633)

    Returns:
        dict: Raw API response with 'current' and 'hourly' keys,
              or None if the request fails.
    """
    url = 'https://api.open-meteo.com/v1/forecast'

    # Request both current conditions and hourly forecast fields.
    # temperature_unit and wind_speed_unit are set to US standard units.
    params = {
        'latitude':  lat,
        'longitude': lon,
        'current': [
            'temperature_2m',    # Current temperature at 2 meters above ground
            'wind_speed_10m',    # Current wind speed at 10 meters above ground
            'weather_code',      # WMO weather code for current conditions
            'precipitation',     # Current precipitation amount in mm
            'visibility'         # Current visibility in meters
        ],
        'hourly': [
            'precipitation_probability',  # % chance of precipitation per hour
            'wind_gusts_10m',             # Wind gust speed per hour in mph
            'weather_code',               # WMO weather code per hour
            'visibility'                  # Visibility per hour in meters
        ],
        'temperature_unit': 'fahrenheit',  # Return temps in °F
        'wind_speed_unit':  'mph',         # Return wind speeds in mph
        'forecast_days': 1                 # Only need today's 24-hour forecast
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # Raises an error for 4xx/5xx HTTP status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        # Catches network errors, timeouts, and bad HTTP status codes
        print(f'  API call failed: {e}')
        return None

print('✅ fetch_weather() defined!')

In [ ]:
def estimate_weather_window(hourly: dict, current_code: int) -> tuple:
    """
    Scans the 24-hour hourly forecast to estimate when severe weather
    starts, how long it lasts, and when conditions are expected to clear.
    This helps the driver plan how long to keep their vehicle in covered parking.

    Parameters:
        hourly (dict)      : The 'hourly' section from the Open-Meteo API response.
                             Contains 'weather_code' and 'time' as 24-element lists.
        current_code (int) : The current WMO weather code.

    Returns:
        tuple: (starts_in_hours, duration_hours, clears_by)
            - starts_in_hours (int)  : Hours until bad weather begins. 0 = already active.
            - duration_hours  (int)  : Consecutive hours of severe weather.
            - clears_by       (str)  : Formatted time when conditions improve.
    """
    codes = hourly['weather_code']  # 24-element list, one entry per hour
    times = hourly['time']          # Corresponding ISO timestamps e.g. '2026-03-31T14:00'

    # ── Case 1: Severe weather is already active right now ──
    if current_code in ALERT_THRESHOLD_CODES:
        starts_in_hours = 0  # Already started

        # Count how many consecutive hours from the start of the forecast
        # remain in a severe state (represents remaining duration)
        duration = 0
        for code in codes:
            if code in ALERT_THRESHOLD_CODES:
                duration += 1
            else:
                break  # Stop at the first non-severe hour

        duration_hours = max(duration, 1)  # At least 1 hour if currently active

    # ── Case 2: Currently clear, scan ahead for incoming severe weather ──
    else:
        starts_in_hours = -1  # Sentinel value: no severe weather found yet

        # Find the first hour in the forecast that has a severe weather code
        for i, code in enumerate(codes):
            if code in ALERT_THRESHOLD_CODES:
                starts_in_hours = i  # e.g. index 2 means severe weather in 2 hours
                break

        # If no severe weather found in the entire 24-hour window, return safe result
        if starts_in_hours == -1:
            return (None, 0, 'No severe weather in forecast')

        # Count consecutive severe hours starting from when it begins
        duration_hours = 0
        for code in codes[starts_in_hours:]:
            if code in ALERT_THRESHOLD_CODES:
                duration_hours += 1
            else:
                break  # Stop at the first non-severe hour after the storm

        duration_hours = max(duration_hours, 1)  # Ensure at least 1 hour

    # ── Compute the index in the hourly list where conditions clear ──
    # clear_index = start of storm + duration = first clear hour after storm
    start_idx   = starts_in_hours if (starts_in_hours and starts_in_hours >= 0) else 0
    clear_index = start_idx + duration_hours

    # Convert the clear hour's timestamp to a readable format like '05:00 PM'
    if clear_index < len(times):
        clear_dt  = datetime.strptime(times[clear_index], '%Y-%m-%dT%H:%M')
        clears_by = clear_dt.strftime('%I:%M %p')
    else:
        # Storm extends beyond the 24-hour forecast window
        clears_by = 'Extends beyond forecast window'

    return (starts_in_hours if starts_in_hours != -1 else 0, duration_hours, clears_by)

print('✅ estimate_weather_window() defined!')

In [ ]:
def parse_weather_alert(location: dict, response: dict) -> WeatherAlert:
    """
    Takes a raw Open-Meteo API response and converts it into a
    structured WeatherAlert object with severity, advice, and
    weather duration information.

    Parameters:
        location (dict) : Dict with 'name', 'lat', 'lon' keys.
        response (dict) : Raw JSON from fetch_weather() or build_mock_response().

    Returns:
        WeatherAlert: Fully populated alert object ready for display.
    """
    current = response['current']  # Current weather snapshot
    hourly  = response['hourly']   # Hourly forecast for the next 24 hours

    # ── Extract current conditions ──
    weather_code   = current['weather_code']
    wind_speed_mph = current['wind_speed_10m']       # Already in mph (requested above)
    visibility_mi  = current['visibility'] / 1609    # Convert meters → miles
    temperature_f  = current['temperature_2m']       # Already in °F
    condition_now  = SEVERE_WEATHER_CODES.get(weather_code, 'Unknown')
    is_severe_now  = weather_code in ALERT_THRESHOLD_CODES

    # ── Check the next 1 hour in the hourly forecast for severe conditions ──
    # We use [:1] to look only 1 hour ahead. Increase to [:3] for 3-hour lookahead.
    upcoming_alerts = []
    for i, code in enumerate(hourly['weather_code'][:1]):
        if code in ALERT_THRESHOLD_CODES:
            upcoming_alerts.append({
                'hour':       hourly['time'][i],
                'condition':  SEVERE_WEATHER_CODES.get(code, 'Unknown'),
                'rain_prob':  hourly['precipitation_probability'][i],
                'wind_gusts': hourly['wind_gusts_10m'][i],
            })

    # ── Determine if an alert should be triggered ──
    # Alert fires if either the current conditions OR the next hour forecast is severe
    alert_required = is_severe_now or len(upcoming_alerts) > 0

    # ── Determine severity level ──
    # Use the current code if severe now, otherwise use the first upcoming severe code
    severity = 'NONE'
    if is_severe_now:
        severity = ALERT_SEVERITY.get(weather_code, 'MEDIUM')
    elif upcoming_alerts:
        severity = ALERT_SEVERITY.get(hourly['weather_code'][0], 'MEDIUM')

    # ── Estimate weather duration window ──
    # Returns when storm starts, how long it lasts, and when it clears
    starts_in_hours, duration_hours, clears_by = estimate_weather_window(hourly, weather_code)

    # ── Pick the right parked vehicle advice ──
    # If currently severe, use current code; otherwise use the first incoming severe code
    advice_code    = weather_code if is_severe_now else (hourly['weather_code'][0] if upcoming_alerts else 0)
    vehicle_advice = PARKED_VEHICLE_ADVICE.get(advice_code, '')

    # ── Build the human-readable alert message ──
    if alert_required:
        if is_severe_now:
            # Storm is already here — tell driver how long it will last
            timing_str    = f'Expected to clear by {clears_by} (~{duration_hours} hr(s))'
            alert_message = (
                f'SAFE PILOT ALERT [{severity}]: {condition_now} at your location now! '
                f'Wind: {wind_speed_mph:.1f} mph | Visibility: {visibility_mi:.1f} mi. '
                f'{timing_str}.'
            )
        else:
            # Storm is incoming — tell driver when it arrives and how long it lasts
            u          = upcoming_alerts[0]
            timing_str = (
                f'Arrives in ~{starts_in_hours} hr(s), lasts ~{duration_hours} hr(s), '
                f'clears by {clears_by}'
            )
            alert_message = (
                f"SAFE PILOT ALERT [{severity}]: {u['condition']} approaching! "
                f"Rain: {u['rain_prob']}% | Gusts: {u['wind_gusts']:.1f} mph. "
                f'{timing_str}.'
            )
    else:
        alert_message = 'No alert needed. Weather conditions are safe.'

    # ── Assemble and return the WeatherAlert object ──
    return WeatherAlert(
        location_name  = location['name'],
        lat            = location['lat'],
        lon            = location['lon'],
        is_severe_now  = is_severe_now,
        condition_now  = condition_now,
        weather_code   = weather_code,
        wind_speed_mph = wind_speed_mph,
        visibility_miles = visibility_mi,
        temperature_f  = temperature_f,
        upcoming_alerts  = upcoming_alerts,
        alert_required = alert_required,
        severity       = severity,
        alert_message  = alert_message,
        vehicle_advice = vehicle_advice,
        starts_in_hours  = starts_in_hours or 0,
        duration_hours   = duration_hours,
        clears_by        = clears_by,
    )

print('✅ parse_weather_alert() defined!')

In [ ]:
def print_alert(alert: WeatherAlert):
    """
    Prints a formatted Safe Pilot weather alert summary to the console.
    Shows current conditions, upcoming forecast, alert message,
    vehicle-specific advice, and the weather duration timeline.

    Parameters:
        alert (WeatherAlert): A populated WeatherAlert object from parse_weather_alert().
    """
    print('=' * 68)
    print(f'  📍 {alert.location_name}')
    print(f'     Lat: {alert.lat} | Lon: {alert.lon}')
    print('-' * 68)

    # ── Current conditions block ──
    print(f'  🌡️  Temperature   : {alert.temperature_f:.1f} F')
    print(f'  🌤️  Condition Now : {alert.condition_now} (code {alert.weather_code})')
    print(f'  💨  Wind Speed    : {alert.wind_speed_mph:.1f} mph')
    print(f'  👁️  Visibility    : {alert.visibility_miles:.1f} miles')

    # ── Upcoming forecast block (only shown if severe weather detected in next hour) ──
    if alert.upcoming_alerts:
        print('\n  📅 Upcoming (next 1 hour):')
        for a in alert.upcoming_alerts:
            print(f"     • {a['hour']} → {a['condition']} | Rain: {a['rain_prob']}% | Gusts: {a['wind_gusts']:.1f} mph")

    # ── Alert message, advice, and timeline ──
    if alert.alert_required:
        print(f'\n  🚨 {alert.alert_message}')

        # Parked vehicle specific action advice
        if alert.vehicle_advice:
            print(f'  💡 Action : {alert.vehicle_advice}')

        # Weather timeline — helps the driver plan how long to stay in covered parking
        print(f'\n  ⏱️  Weather Timeline:')
        if alert.is_severe_now:
            print(f'     • Status    : Active NOW')  # Storm already here
        else:
            print(f'     • Starts in : ~{alert.starts_in_hours} hour(s)')  # Incoming storm
        print(f'     • Duration  : ~{alert.duration_hours} hour(s)')
        print(f'     • Clears by : {alert.clears_by}')  # When it is safe to return
    else:
        # No alert — conditions are safe
        print(f'\n  ✅ {alert.alert_message}')

    print('=' * 68)
    print()

print('✅ print_alert() defined!')

In [ ]:
def check_weather_at_location(location: dict):
    """
    Main entry point for a single location check.
    Orchestrates the full pipeline:
      1. fetch_weather()        → calls Open-Meteo API with lat/lon
      2. parse_weather_alert()  → converts raw response into WeatherAlert
      3. print_alert()          → displays formatted output

    Parameters:
        location (dict): Must have 'name', 'lat', and 'lon' keys.

    Returns:
        WeatherAlert or None if the API call failed.
    """
    print(f"\n🔍 Checking: {location['name']} ...")

    # Step 1: Fetch live weather data from Open-Meteo
    response = fetch_weather(location['lat'], location['lon'])

    # If API call failed (network issue, timeout, etc.), skip this location
    if response is None:
        print('  ❌ Could not retrieve weather data.')
        return None

    # Step 2: Parse the response into a structured alert object
    alert = parse_weather_alert(location, response)

    # Step 3: Print the formatted alert to the console
    print_alert(alert)

    return alert

print('✅ check_weather_at_location() defined!')

---
## 🌍 Section A — Real Live Weather (5 Locations)
Results depend on actual current weather conditions at time of run.

In [ ]:
# ── 5 example GPS locations across different US climate zones ──
# These represent real coordinates where Safe Pilot could be deployed.
# Alert results will vary based on live weather at time of execution.
REAL_LOCATIONS = [
    {'name': 'Edinburg, TX',       'lat': 26.3017,  'lon': -98.1633},   # South Texas — subtropical
    {'name': 'Miami, FL',          'lat': 25.7617,  'lon': -80.1918},   # Florida — storm-prone
    {'name': 'Denver, CO',         'lat': 39.7392,  'lon': -104.9903},  # Colorado — mountain weather
    {'name': 'Oklahoma City, OK',  'lat': 35.4676,  'lon': -97.5164},   # Tornado Alley
    {'name': 'San Diego, CA',      'lat': 32.7157,  'lon': -117.1611},  # California — usually clear
]

print('\n' + '🛡️  SAFE PILOT — SECTION A: Live Weather'.center(68))
print(f"{'Run at: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'):^68}\n")

alerts_triggered, no_alerts = [], []

# Loop through each location, fetch live weather, and evaluate alerts
for location in REAL_LOCATIONS:
    alert = check_weather_at_location(location)
    if alert:
        # Bucket results into alert vs no-alert for the summary
        (alerts_triggered if alert.alert_required else no_alerts).append(alert.location_name)

# ── Print summary of all locations ──
print('\n' + '-' * 68)
print('📊 SECTION A SUMMARY')
print('-' * 68)
if alerts_triggered:
    print(f'  🚨 Alerts Triggered ({len(alerts_triggered)}):')
    for name in alerts_triggered: print(f'     • {name}')
if no_alerts:
    print(f'  ✅ No Alert Needed ({len(no_alerts)}):')
    for name in no_alerts: print(f'     • {name}')
print('-' * 68)

---
## 🧪 Section B — Simulated Scenarios (Parked Vehicle)

These scenarios **inject specific weather codes** into a real API response so the demo
always produces predictable alert output regardless of live weather.

**Group 1 — Currently CLEAR, severe weather incoming:**

| Scenario | Current | Incoming | Severity |
|---|---|---|---|
| 1 | Clear Sky | Thunderstorm + Heavy Hail in 1h | 🔴 CRITICAL |
| 2 | Clear Sky | Thunderstorm + Hail in 2h | 🔴 HIGH |
| 3 | Partly Cloudy | Heavy Snow in 2h | 🟠 HIGH |
| 4 | Overcast | Heavy Rain in 1h | 🟡 MEDIUM |

**Group 2 — Currently SEVERE weather active:**

| Scenario | Current | Severity |
|---|---|---|
| 5 | Thunderstorm + Heavy Hail | 🔴 CRITICAL |
| 6 | Thunderstorm | 🔴 HIGH |
| 7 | Violent Rain Showers | 🟠 HIGH |
| 8 | Icy Fog | 🟡 MEDIUM |

**Group 3 — All clear:**

| Scenario | Current | Severity |
|---|---|---|
| 9 | Clear Sky | ✅ NONE |

In [ ]:
def build_mock_response(base_response: dict,
                         current_code: int,
                         current_wind_mph: float,
                         current_visibility_m: float,
                         hourly_codes: list) -> dict:
    """
    Clones a real Open-Meteo API response and injects specific
    weather codes and conditions for simulation/demo purposes.

    This allows testing every alert scenario (e.g. hail, thunderstorm,
    snow) without needing those conditions to actually occur.

    Parameters:
        base_response         (dict)  : A real API response to clone as the base structure.
        current_code          (int)   : WMO code to set as current weather (e.g. 0 = clear, 99 = hail).
        current_wind_mph      (float) : Wind speed to inject for current conditions.
        current_visibility_m  (float) : Visibility in meters to inject for current conditions.
        hourly_codes          (list)  : List of up to 24 WMO codes for the hourly forecast.
                                        Index 0 = next hour, index 1 = 2 hours from now, etc.

    Returns:
        dict: A cloned and modified API response ready to pass into parse_weather_alert().
    """
    # deepcopy prevents modifying the original base_response across multiple scenario runs
    mock = deepcopy(base_response)

    # ── Override current weather conditions ──
    mock['current']['weather_code']   = current_code
    mock['current']['wind_speed_10m'] = current_wind_mph
    mock['current']['visibility']     = current_visibility_m

    # ── Override hourly forecast with scenario-specific codes ──
    # Also inject realistic rain probability and wind gust values
    # based on the severity of each hour's weather code
    n = len(hourly_codes)
    for i in range(min(n, 24)):  # Cap at 24 hours (API max)
        code = hourly_codes[i]
        mock['hourly']['weather_code'][i] = code

        # Assign rain probability and gusts based on severity tier
        if code in {99}:             # Thunderstorm + Heavy Hail
            mock['hourly']['precipitation_probability'][i] = 95
            mock['hourly']['wind_gusts_10m'][i]            = 82.0
        elif code in {96, 95}:       # Thunderstorm / Thunderstorm + Hail
            mock['hourly']['precipitation_probability'][i] = 85
            mock['hourly']['wind_gusts_10m'][i]            = 62.0
        elif code in {82, 75}:       # Violent Showers / Heavy Snow
            mock['hourly']['precipitation_probability'][i] = 75
            mock['hourly']['wind_gusts_10m'][i]            = 45.0
        elif code in {65, 48}:       # Heavy Rain / Icy Fog
            mock['hourly']['precipitation_probability'][i] = 60
            mock['hourly']['wind_gusts_10m'][i]            = 22.0
        else:                        # Clear / non-severe conditions
            mock['hourly']['precipitation_probability'][i] = 5
            mock['hourly']['wind_gusts_10m'][i]            = 8.0

    return mock

print('✅ build_mock_response() defined!')

In [ ]:
# ── Fetch one real API response to use as the base for all mock scenarios ──
# We only need the structure (timestamps, etc.) — the weather values will be overridden.
BASE_LOCATION = {'name': 'Edinburg, TX', 'lat': 26.3017, 'lon': -98.1633}
print('Fetching base weather response for simulation...')
base_response = fetch_weather(BASE_LOCATION['lat'], BASE_LOCATION['lon'])

if base_response is None:
    print('Could not fetch base response. Check internet connection.')
else:
    # ──────────────────────────────────────────────────────────────────────
    # SIMULATED SCENARIOS
    # Each scenario defines:
    #   label                : Description shown in output
    #   current_code         : WMO code for conditions RIGHT NOW
    #   current_wind_mph     : Wind speed now in mph
    #   current_visibility_m : Visibility now in meters
    #   hourly_codes         : 24-element list of WMO codes (index = hour offset)
    #                          e.g. index 2 = conditions 2 hours from now
    # ──────────────────────────────────────────────────────────────────────
    SIMULATED_SCENARIOS = [

        # ── GROUP 1: Currently CLEAR — severe weather incoming ─────────────

        {
            # Clear now, heavy hail storm arrives in 1 hour and lasts 3 hours
            'label': 'Scenario 1 | CLEAR NOW → Thunderstorm + Heavy Hail in 1h [CRITICAL]',
            'current_code': 0, 'current_wind_mph': 5.0, 'current_visibility_m': 24000,
            'hourly_codes': [0, 99, 99, 99, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Clear now, hail storm arrives in 2 hours and lasts 2 hours
            'label': 'Scenario 2 | CLEAR NOW → Thunderstorm + Hail in 2h [HIGH]',
            'current_code': 0, 'current_wind_mph': 7.0, 'current_visibility_m': 22000,
            'hourly_codes': [0, 0, 96, 96, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Partly cloudy now, heavy snow arrives in 2 hours and lasts 5 hours
            'label': 'Scenario 3 | PARTLY CLOUDY → Heavy Snow in 2h [HIGH]',
            'current_code': 2, 'current_wind_mph': 12.0, 'current_visibility_m': 18000,
            'hourly_codes': [2, 2, 75, 75, 75, 75, 75, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Overcast now, heavy rain arrives in 1 hour and lasts 4 hours, then gradually clears
            'label': 'Scenario 4 | OVERCAST → Heavy Rain in 1h [MEDIUM]',
            'current_code': 3, 'current_wind_mph': 15.0, 'current_visibility_m': 12000,
            'hourly_codes': [3, 65, 65, 65, 65, 63, 61, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },

        # ── GROUP 2: Currently SEVERE — storm active now ────────────────────

        {
            # Heavy hail storm active now, lasts 2 more hours then gradually clears
            'label': 'Scenario 5 | Thunderstorm + Heavy Hail ACTIVE [CRITICAL]',
            'current_code': 99, 'current_wind_mph': 68.0, 'current_visibility_m': 800,
            'hourly_codes': [99, 99, 96, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Thunderstorm active now, lasts 3 hours, then tapering rain
            'label': 'Scenario 6 | Thunderstorm ACTIVE [HIGH]',
            'current_code': 95, 'current_wind_mph': 43.0, 'current_visibility_m': 2500,
            'hourly_codes': [95, 95, 95, 82, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Violent rain active now, lasts 2 hours, then moderate/light rain
            'label': 'Scenario 7 | Violent Rain Showers ACTIVE [HIGH]',
            'current_code': 82, 'current_wind_mph': 38.0, 'current_visibility_m': 3000,
            'hourly_codes': [82, 82, 65, 63, 61, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
        {
            # Icy fog active now and persisting for 4 hours, then regular fog then clear
            'label': 'Scenario 8 | Icy Fog ACTIVE [MEDIUM]',
            'current_code': 48, 'current_wind_mph': 8.0, 'current_visibility_m': 200,
            'hourly_codes': [48, 48, 48, 48, 45, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },

        # ── GROUP 3: All clear — no alert ──────────────────────────────────

        {
            # Perfectly clear now with no severe weather in the 24-hour forecast
            'label': 'Scenario 9 | Clear Sky — No Alert',
            'current_code': 0, 'current_wind_mph': 5.0, 'current_visibility_m': 24000,
            'hourly_codes': [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        },
    ]

    print('\n' + '🧪  SAFE PILOT — SECTION B: Simulated Scenarios'.center(68) + '\n')

    alerts_triggered, no_alerts = [], []

    # ── Run each scenario through the full parse + print pipeline ──
    for s in SIMULATED_SCENARIOS:
        # Build a mock response by injecting this scenario's codes into the base response
        mock = build_mock_response(
            base_response,
            current_code         = s['current_code'],
            current_wind_mph     = s['current_wind_mph'],
            current_visibility_m = s['current_visibility_m'],
            hourly_codes         = s['hourly_codes'],
        )
        loc = {'name': s['label'], 'lat': BASE_LOCATION['lat'], 'lon': BASE_LOCATION['lon']}
        print(f"\n🔍 Running: {s['label']}")
        alert = parse_weather_alert(loc, mock)
        print_alert(alert)
        (alerts_triggered if alert.alert_required else no_alerts).append(s['label'])

    # ── Section B summary ──
    print('\n' + '-' * 68)
    print('📊 SECTION B SUMMARY')
    print('-' * 68)
    if alerts_triggered:
        print(f'  🚨 Alerts Triggered ({len(alerts_triggered)}):')
        for name in alerts_triggered: print(f'     • {name}')
    if no_alerts:
        print(f'  ✅ No Alert Needed ({len(no_alerts)}):')
        for name in no_alerts: print(f'     • {name}')
    print('-' * 68)

---
## ✏️ Section C — Test Your Own Lat/Lon
Replace the lat/lon values below with any GPS coordinate to check live weather.

In [ ]:
# ── Custom location test ──
# Replace lat and lon with your own GPS coordinates to test any location.
# Tip: Right-click any location on Google Maps → click the coordinates to copy them.
custom_location = {
    'name': 'My Custom Location',
    'lat':  26.3017,   # Replace with your latitude  (e.g. 40.7128 for New York)
    'lon': -98.1633    # Replace with your longitude (e.g. -74.0060 for New York)
}

check_weather_at_location(custom_location)